# Covalent Protein Modification with mBuild + OpenFF Pablo

This notebook builds a maximally modified ubiquitin (1UBQ) with **mBuild** as the coordinate generator, and loads the product with **OpenFF Pablo** for parameterization:

1. **LYS63 NZ** — an SBM–NHS–EGP methacrylate trimer, amide-bonded through the **middle** (NHS-derived) monomer. *A multi-residue linear polymer.*
2. **ASN60 ND2** — a chitobiose N-glycan loaded from a **GLYCAM PDB with CONECT records**. *A multi-residue glycan from PDB.*
3. **SER20 OG** — a three-branch sugar from **SMILES** as one residue. *A branched glycan from SMILES.*

The handoff artifact is a plain PDB file plus small crosslink specs. mBuild carries **no OpenFF dependency**; all OpenFF code lives in this notebook.

**Requirements**: mBuild (`biopolymers` feature branches), RDKit, `openff-pablo >= 0.2`, `openff-toolkit`, and (once) `pdbfixer` to protonate the input.

**Fragment sources, in order of preference**
- **SMILES / SDF** — chemistry-complete (bond orders, formal charges; `fragment_from_sdf` reads SDF). Author the *product* fragment: for NHS chemistry, the acyl fragment as it appears after conjugation, with one H at the future bond site. The NHS leaving group never enters mBuild.
- **PDB + CONECT** — legacy fallback (`fragment_from_pdb`). CONECT records carry no bond orders, so multiple bonds must be declared explicitly (`bond_orders=...`).

The protein input must be **fully protonated** at the target pH (pdbfixer or reduce). The loader errors, with the residue named, on anything it cannot match — it never guesses chemistry.


In [1]:
# One-time preparation: protonate the crystal structure at pH 7.
# Run in an environment with pdbfixer + openmm, then reuse the file.
import os

if not os.path.exists("1ubq_protonated.pdb"):
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile

    fixer = PDBFixer(filename="1UBQ_testProtein.cleaned.pdb")
    fixer.findMissingResidues(); fixer.missingResidues = {}
    fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms(); fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    with open("1ubq_protonated.pdb", "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle)


In [2]:
import numpy as np
import mbuild as mb
from mbuild.biopolymers import Protein, prepare_fragment
from mbuild_extras import fragment_from_pdb, pablo_crosslink_kwargs

DEMO = "."
GLYCAN_PDB = "glycam_G42666HT_CONECT.pdb"

# Product-form monomer units: backbone written as CH3-C(CH3)(R)-H, so one
# hydrogen on each backbone carbon can leave when the chain links.
SBM_SMILES = "CC(C)C(=O)OCC[N+](C)(C)CCCS(=O)(=O)[O-]"
EGP_SMILES = "CC(C)C(=O)OCCOc1ccccc1"
NHS_SMILES = "CC(C)C=O"  # the aldehyde H marks the future amide bond

THREEBRANCH_SMILES = (
    "CC(=O)N[C@@H]1[C@H]([C@@H]([C@H](O[CH2]1)CO)"  # anomeric OH -> CH2
    "O[C@H]2[C@@H]([C@H]([C@@H]([C@H](O2)CO)"
    "O[C@H]3[C@H]([C@H]([C@@H]([C@H](O3)CO[C@@H]4[C@H]([C@H]([C@@H]([C@H](O4)"
    "CO[C@@H]5[C@H]([C@H]([C@@H]([C@H](O5)CO)O)O)O)O)"
    "O[C@@H]6[C@H]([C@H]([C@@H]([C@H](O6)CO)O)O)"
    "O[C@@H]7[C@H]([C@H]([C@@H]([C@H](O7)CO)O)O)O)O)O)"
    "O[C@@H]8[C@H]([C@H]([C@@H]([C@H](O8)CO)O)O)"
    "O[C@@H]9[C@H]([C@H]([C@@H]([C@H](O9)CO)O)O)"
    "O[C@@H]1[C@H]([C@H]([C@@H]([C@H](O1)CO)O)O)O)O)O)NC(=O)C)O"
)


## Helpers

`prepare_fragment` returns a fragment as a named `Residue` with **final atom names** — read the names, pick the attachment atom, and reuse the same names in the Pablo definitions. The two backbone carbons of every monomer get fixed names (`CBH`, `CBT`) so all monomers share one Pablo *linking bond*.


In [3]:
def prepare_monomer(smiles, resname):
    """Prepare a monomer unit with fixed backbone atom names CBH/CBT."""
    residue = prepare_fragment(mb.load(smiles, smiles=True), resname)
    particles = list(residue.particles())
    particles[0].name = "CBH"  # head backbone carbon (CH3 in the free unit)
    particles[1].name = "CBT"  # tail backbone carbon (carries R and one H)
    return residue


def carbonyl_carbon(residue):
    """Return the name of the carbon double-bonded to an oxygen."""
    root = residue.root if residue.parent is not None else residue
    for p1, p2, data in root.bonds(return_bond_order=True):
        if data["bond_order"] == 2.0 and {p1.element.symbol, p2.element.symbol} == {"C", "O"}:
            return (p1 if p1.element.symbol == "C" else p2).name
    raise ValueError("no carbonyl found")


def anomeric_ch2(residue):
    """Return the ring CH2 bonded to the ring oxygen (the attachment C)."""
    for particle in residue.particles():
        if particle.element.symbol != "C":
            continue
        neighbors = list(particle.direct_bonds())
        hydrogens = [n for n in neighbors if n.element.symbol == "H"]
        oxygens = [n for n in neighbors if n.element.symbol == "O"]
        carbons = [n for n in neighbors if n.element.symbol == "C"]
        if len(hydrogens) == 2 and len(oxygens) == 1 and len(carbons) == 1:
            ring_oxygen = oxygens[0]
            heavy = [
                n for n in ring_oxygen.direct_bonds() if n.element.symbol != "H"
            ]
            if len(heavy) == 2:
                return particle.name
    raise ValueError("no anomeric CH2 found")


## Load the protein

Template matching against the bundled CCD library stamps full chemistry: bonds with orders, formal charges, charged termini, histidine tautomers.


In [4]:
protein = Protein(f"{DEMO}/1ubq_protonated.pdb")
print("loaded:", protein.n_particles, "atoms, net", protein.net_formal_charge)


loaded: 1231 atoms, net 0


## Modification 1 — polymer trimer at LYS63 (amide through the middle monomer)

`attach()` substitutes one hydrogen on each side and records the bond with its leaving hydrogens. For a true **amide**, the lysine loses a second proton (`HZ3`), so Pablo later matches its *neutral* lysine variant and the amide nitrogen is uncharged.

The chain is built by repeated `attach()` calls — an attached residue is addressable like any other. Residue **numbers set the backbone order in the file**, which is what Pablo's adjacency-based polymer linking reads.


In [5]:
# --- 1. trimer at LYS63, reactive monomer in the middle ---------------
sbm_unit = prepare_monomer(SBM_SMILES, "SBM")
egp_unit = prepare_monomer(EGP_SMILES, "EGP")
nhs_unit = prepare_monomer(NHS_SMILES, "NHS")
nhs_site = carbonyl_carbon(nhs_unit)
print("NHS carbonyl atom:", nhs_site)

rec_lys = protein.attach(nhs_unit, nhs_site, resnum=63, atom_name="NZ",
                         chain_id="A", fragment_resname="NHS")
# Amide nitrogen is neutral: remove a second proton from NZ.
lys63 = protein.get_residue(63, chain_id="A")
hz3 = [p for p in lys63.particles() if p.name == "HZ3"][0]
protein.remove(hz3)
lys63.formal_charge -= 1

nhs_res = rec_lys.residue2
rec_sbm = protein.attach(sbm_unit, "CBT", resnum=nhs_res.resnum,
                         atom_name="CBH", chain_id="A")
rec_egp = protein.attach(egp_unit, "CBH", resnum=nhs_res.resnum,
                         atom_name="CBT", chain_id="A")
# Backbone order in the file must be SBM-NHS-EGP for Pablo adjacency.
rec_sbm.residue2.resnum, nhs_res.resnum, rec_egp.residue2.resnum = 77, 78, 79
print("trimer records:", [(r.atom1_name, r.atom2_name, r.leaving1, r.leaving2)
                          for r in (rec_lys, rec_sbm, rec_egp)])


NHS carbonyl atom: C4
2026-08-28 14:09:08,583 - mbuild.biopolymers.protein - WARNING - 2 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.81 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


2026-08-28 14:09:10,746 - mbuild.biopolymers.protein - WARNING - 1 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.43 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


trimer records: [('NZ', 'C4', ('HZ1',), ('H8',)), ('CBH', 'CBT', ('H1',), ('H4',)), ('CBT', 'CBH', ('H4',), ('H1',))]


## Modification 2 — chitobiose from a GLYCAM PDB at ASN60

Bonds come only from CONECT records; the two N-acetyl `C=O` double bonds are declared explicitly. The GLYCAM `ROH` cap is the anomeric hydroxyl — it leaves on conjugation, so it is swapped for a placeholder hydrogen `H0` that `attach()` then substitutes. The second sugar's anomeric carbon is renamed `C1A` so the inter-sugar linking bond (`O4 -> C1A`) cannot be confused with the crosslink atom `C1` of the first sugar.


In [6]:
# --- 2. chitobiose from GLYCAM PDB at ASN60 ---------------------------
glycan = fragment_from_pdb(
    GLYCAN_PDB,
    bond_orders={
        ((2, "C2N"), (2, "O2N")): 2,  # N-acetyl C=O of 4YB
        ((3, "C2N"), (3, "O2N")): 2,  # N-acetyl C=O of 0YB
    },
)
resnames = [r.name for r in glycan.children]
print("glycan residues:", resnames)
# The GLYCAM ROH cap is the anomeric OH; it leaves on conjugation.
roh = [r for r in glycan.children if r.name == "ROH"][0]
fyb = [r for r in glycan.children if r.name == "4YB"][0]
c1 = [p for p in fyb.particles() if p.name == "C1"][0]
o1 = [p for p in roh.particles() if p.name == "O1"][0]
direction = (o1.pos - c1.pos) / np.linalg.norm(o1.pos - c1.pos)
glycan.remove(list(roh.particles()))
placeholder = mb.Compound(name="H0", element="H", pos=c1.pos + 0.109 * direction)
fyb.add(placeholder)
glycan.add_bond((c1, placeholder), bond_order=1.0)
# Rename the second sugar's anomeric carbon so the inter-sugar linking
# bond (O4 -> C1A) cannot be confused with 4YB's crosslink atom C1.
oyb = [r for r in glycan.children if r.name == "0YB"][0]
[p for p in oyb.particles() if p.name == "C1"][0].name = "C1A"

rec_glycan = protein.attach(glycan, "C1", resnum=60, atom_name="ND2",
                            chain_id="A", fragment_resnum=2)
print("glycan record:", rec_glycan.atom1_name, rec_glycan.atom2_name,
      rec_glycan.leaving1, rec_glycan.leaving2)


glycan residues: ['ROH', '4YB', '0YB']
glycan record: ND2 C1 ('HD21',) ('H0',)


## Modification 3 — three-branch glycan from SMILES at SER20

The SMILES is authored in product form: the anomeric hydroxyl is already replaced by the CH2 that bonds to the serine oxygen. Branching **inside one residue** is unrestricted.


In [7]:
# --- 3. three-branch glycan from SMILES at SER20 ----------------------
ng3 = prepare_fragment(mb.load(THREEBRANCH_SMILES, smiles=True), "NG3")
ng3_site = anomeric_ch2(ng3)
rec_ng3 = protein.attach(ng3, ng3_site, resnum=20, atom_name="OG",
                         chain_id="A")
print("NG3 site:", ng3_site, "record:", rec_ng3.leaving1, rec_ng3.leaving2)


2026-08-28 14:09:17,238 - mbuild.biopolymers.protein - WARNING - 11 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.54 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


NG3 site: C7 record: ('HG',) ('H10',)


## Export: the handoff artifact

`save_pdb` writes a Pablo-conformant PDB (residues ordered by number, `TER` per chain, `CONECT` only for bonds adjacency does not explain). `bond_records()` returns neutral records for every recorded bond; the glue formats them into `with_crosslink` kwargs, and the polymer **backbone** bonds ride on Pablo's linking mechanism instead, so they are filtered out. The warning about NHS is expected — see the closing notes.


In [8]:
protein.save_pdb(f"{DEMO}/1ubq_modified.pdb", overwrite=True)
records = protein.bond_records()
crosslink_only = [
    pablo_crosslink_kwargs(record)
    for record in records
    if set(record["atom_names"]) != {"CBH", "CBT"}  # backbone rides linking
]
print("specs for with_crosslink:", crosslink_only)
print("net formal charge:", protein.net_formal_charge)


specs for with_crosslink: [{'residues': ['LYS', 'NHS'], 'linking_atoms': ['NZ', 'C4'], 'leaving_atoms': [['HZ1'], ['H8']], 'bond_order': 1}, {'residues': ['ASN', '4YB'], 'linking_atoms': ['ND2', 'C1'], 'leaving_atoms': [['HD21'], ['H0']], 'bond_order': 1}, {'residues': ['SER', 'NG3'], 'linking_atoms': ['OG', 'C7'], 'leaving_atoms': [['HG'], ['H10']], 'bond_order': 1}]
net formal charge: -1


## Pablo side: residue definitions for the custom fragments

One **named** `ResidueDefinition` per fragment residue type, built from the same chemistry source the fragment came from (SMILES, or the fragment graph for the GLYCAM sugars). Two mechanisms carry the links:
- **Linking bonds** (`CBT -> CBH` for the polymer backbone, `O4 -> C1A` for the sugars): formed by residue adjacency, any number per residue.
- **Crosslinks** (`with_crosslink(**spec)`): formed via CONECT records; Pablo 0.2.2 allows **one per residue definition**.


In [9]:
from rdkit import Chem
from openff.toolkit import Molecule
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb
from openff.pablo.residue import BondDefinition

POLYMER_LINK = BondDefinition.with_defaults("CBT", "CBH")
GLYCO_LINK = BondDefinition.with_defaults("O4", "C1A")


def named_offmol_from_smiles(smiles, prepared_residue):
    """OpenFF molecule from SMILES with mBuild's final atom names."""
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    offmol = Molecule.from_rdkit(
        rdmol, allow_undefined_stereo=True, hydrogens_are_explicit=True
    )
    particles = list(prepared_residue.particles())
    assert [a.symbol for a in offmol.atoms] == [
        p.element.symbol for p in particles
    ], "atom order mismatch between SMILES and mBuild fragment"
    for atom, particle in zip(offmol.atoms, particles):
        atom.name = particle.name
    return offmol


def monomer_definition(smiles, prepared_residue, resname, leaving_names):
    offmol = named_offmol_from_smiles(smiles, prepared_residue)
    for atom in offmol.atoms:
        if atom.name in leaving_names:
            atom.metadata["leaving_atom"] = True
    return ResidueDefinition.from_molecule(
        offmol, residue_name=resname, linking_bond=POLYMER_LINK
    )


def sugar_definition(residue, resname, extra_hydrogens, linking_bond):
    """Definition from an mBuild fragment residue graph plus leaving Hs.

    Sugars from the GLYCAM PDB are neutral; bond orders come from the
    fragment (single, plus the declared C=O bonds).
    """
    particles = list(residue.particles())
    editable = Chem.RWMol()
    index = {}
    for particle in particles:
        atom = Chem.Atom(particle.element.symbol)
        atom.SetNoImplicit(True)
        index[particle] = editable.AddAtom(atom)
    root = residue.root
    for p1, p2, data in root.bonds(return_bond_order=True):
        if p1 in index and p2 in index:
            bond_type = (
                Chem.BondType.DOUBLE
                if data["bond_order"] == 2.0
                else Chem.BondType.SINGLE
            )
            editable.AddBond(index[p1], index[p2], bond_type)
    names = [particle.name for particle in particles]
    by_name = {particle.name: index[particle] for particle in particles}
    for host_name, hydrogen_name in extra_hydrogens:
        atom = Chem.Atom("H")
        atom.SetNoImplicit(True)
        new_index = editable.AddAtom(atom)
        editable.AddBond(by_name[host_name], new_index, Chem.BondType.SINGLE)
        names.append(hydrogen_name)
    mol = editable.GetMol()
    Chem.SanitizeMol(mol)
    offmol = Molecule.from_rdkit(
        mol, allow_undefined_stereo=True, hydrogens_are_explicit=True
    )
    extra_names = {hydrogen for _, hydrogen in extra_hydrogens}
    for atom, name in zip(offmol.atoms, names):
        atom.name = name
        if name in extra_names:
            atom.metadata["leaving_atom"] = True
    return ResidueDefinition.from_molecule(
        offmol, residue_name=resname, linking_bond=linking_bond
    )


In [10]:
sbm_def = monomer_definition(SBM_SMILES, sbm_unit, "SBM", set(rec_sbm.leaving2))
egp_def = monomer_definition(EGP_SMILES, egp_unit, "EGP", set(rec_egp.leaving2))
nhs_def = monomer_definition(
    NHS_SMILES, nhs_unit, "NHS",
    set(rec_sbm.leaving1) | set(rec_egp.leaving1),
)
fyb_def = sugar_definition(fyb, "4YB", [("O4", "HO4")], GLYCO_LINK)
oyb_def = sugar_definition(oyb, "0YB", [("C1A", "HL")], GLYCO_LINK)
ng3_def_mol = named_offmol_from_smiles(THREEBRANCH_SMILES, ng3)
ng3_def = ResidueDefinition.from_molecule(ng3_def_mol, residue_name="NG3")

library = STD_CCD_CACHE.with_(
    {
        "SBM": [sbm_def],
        "NHS": [nhs_def],
        "EGP": [egp_def],
        "4YB": [fyb_def],
        "0YB": [oyb_def],
        "NG3": [ng3_def],
    }
)
for spec in crosslink_only:
    library = library.with_crosslink(**spec)


## Load through Pablo and verify


In [11]:
top = topology_from_pdb(f"{DEMO}/1ubq_modified.pdb", residue_library=library)
mol = top.molecule(0)
print("PABLO OK — molecules:", top.n_molecules, "atoms:", mol.n_atoms,
      "net charge:", mol.total_charge)
nz = [a for a in mol.atoms
      if a.name == "NZ" and a.metadata.get("residue_number") == 63][0]
print("LYS63 NZ charge:", nz.formal_charge, "bonded:",
      sorted(n.name for n in nz.bonded_atoms))
sbm_charges = {a.name: int(a.formal_charge.m)
               for a in mol.atoms
               if a.metadata.get("residue_name") == "SBM" and a.formal_charge.m}
print("SBM charged atoms:", sbm_charges)


PABLO OK — molecules: 1 atoms: 1585 net charge: -1.0 elementary_charge
LYS63 NZ charge: 0 elementary_charge bonded: ['C4', 'CE', 'HZ2']
SBM charged atoms: {'N1': 1, 'O5': -1}


## Next step: parameterize with Interchange

With the OpenFF `Topology` in hand, parameter assignment is standard OpenFF code (choose force fields and charge models per your study; the Rosemary alpha treats proteins and modifications with one model):

```python
from openff.toolkit import ForceField

force_field = ForceField("openff_no_water-3.0.0-alpha0.offxml", "opc3.offxml")
interchange = force_field.create_interchange(top)
simulation = interchange.to_openmm_simulation(...)
```

## Known upstream limits (OpenFF asks)

1. **One crosslink per residue definition** (Pablo 0.2.2). Linear multi-residue fragments work through linking bonds; a *branch-hub residue* with more than one non-adjacent link does not load yet.
2. `with_crosslink` accepts only residue names present in the cache; adding custom named definitions first (as done here) is the workaround.
3. CCD saccharide components lack a linking-type mapping in Pablo, so sugar definitions are custom-built here.

mBuild records every bond without limits, so these are Pablo-side feature requests, not modeling losses.
